In [ ]:
# Generate dummy data for petastorm learning
# This will create parquet files in the data/ directory following the structure:
# data/ds=YYYYMMDD/h=HH/<uuid>.parquet

#!python dummy_data_gen.py --start-date 20260101 --end-date 20260114 --null-probability 0.05

Generating data from 20260101 to 20260114
Rows per file: 500
Files per hour: 4
Null probability: 0.05
Output directory: data

Generated 100 files...
Generated 200 files...
Generated 300 files...
Generated 400 files...
Generated 500 files...
Generated 600 files...
Generated 700 files...
Generated 800 files...
Generated 900 files...
Generated 1000 files...
Generated 1100 files...
Generated 1200 files...
Generated 1300 files...

✅ Successfully generated 1344 parquet files
   Date range: 20260101 to 20260114
   Total dates: 14
   Files per hour: 4
   Rows per file: 500


# PyArrow Dataset Wrapper for PyTorch

Replaces petastorm with modern PyArrow dataset API that scales to many partitions.

**Key advantage**: `ds` and `h` columns are now stored directly in parquet files, eliminating partition column handling complexity.

In [2]:
# Imports
import pyarrow.dataset as ds
import pyarrow as pa
import torch
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path

# Import PyArrowParquetDataset from separate file (required for multi-worker DataLoader)
from pyarrow_dataset import PyArrowParquetDataset


In [2]:
# PyArrowParquetDataset is now imported from pyarrow_dataset.py
# This allows multi-worker DataLoader to work (workers can pickle the class)
#
# See pyarrow_dataset.py for the full implementation
print(f"PyArrowParquetDataset loaded from: pyarrow_dataset.py")
print(f"Features: batch reading, filtering, shuffling, multi-worker support")


PyArrowParquetDataset loaded from: pyarrow_dataset.py
Features: batch reading, filtering, shuffling, multi-worker support


## Basic Usage: Read all data (works with 1344 files instantly!)

In [3]:
# Create dataset - this initializes instantly even with 1344 files!
data_path = Path("data").resolve()
dataset = PyArrowParquetDataset(data_path, batch_size=8)

# Show available dates in dataset
print(f"Dataset schema: {dataset.schema}")
print(f"Number of fragments (parquet files): {len(list(dataset.dataset.get_fragments()))}")

# Create DataLoader
loader = DataLoader(dataset, batch_size=None)  # batch_size=None since dataset already batches

# Get first batch
batch = next(iter(loader))

print("\nBatch keys:", list(batch.keys()))
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {v.shape}, dtype={v.dtype}")

print(f"\nDate in this batch: {batch['ds'][0].item()} (single batch typically comes from one file)")



Dataset schema: ds: int32
h: int32
swiper_id: int64
swipee_id: int64
feat1: double
feat2: double
feat3: double
feat4: double
feat5: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1060
Number of fragments (parquet files): 1344

Batch keys: ['ds', 'h', 'swiper_id', 'swipee_id', 'feat1', 'feat2', 'feat3', 'feat4', 'feat5']

Batch shapes:
  ds: torch.Size([8]), dtype=torch.int32
  h: torch.Size([8]), dtype=torch.int32
  swiper_id: torch.Size([8]), dtype=torch.int64
  swipee_id: torch.Size([8]), dtype=torch.int64
  feat1: torch.Size([8]), dtype=torch.float64
  feat2: torch.Size([8]), dtype=torch.float64
  feat3: torch.Size([8]), dtype=torch.float64
  feat4: torch.Size([8]), dtype=torch.float64
  feat5: torch.Size([8]), dtype=torch.float64

Date in this batch: 20260101 (single batch typically comes from one file)


/Users/fox/Projects/jupyter_notebook_projects/ml_misc/torch_data_pipe_learning/pyarrow_dataset.py:129: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:209.)
  key: torch.as_tensor(value)


## Filtering: Read only specific date range

In [4]:
# Filter to only dates 20260101-20260107
filters = (ds.field("ds") >= 20260101) & (ds.field("ds") <= 20260107)

dataset_filtered = PyArrowParquetDataset(
    data_path, 
    batch_size=8,
    filters=filters
)

# Show how many fragments match the filter
filtered_fragments = list(dataset_filtered.dataset.get_fragments(filter=filters))
print(f"Fragments matching filter (ds 20260101-20260107): {len(filtered_fragments)}")

loader_filtered = DataLoader(dataset_filtered, batch_size=None)

# Collect multiple batches to verify filtering works across dates
all_dates = set()
for i, batch in enumerate(loader_filtered):
    all_dates.update(batch['ds'].unique().tolist())
    if i >= 100:  # Sample first 100 batches
        break

print(f"Unique dates seen in first 100 batches: {sorted(all_dates)}")
print(f"✓ All dates are within filter range [20260101, 20260107]")

Fragments matching filter (ds 20260101-20260107): 672
Unique dates seen in first 100 batches: [20260101]
✓ All dates are within filter range [20260101, 20260107]


## Shuffling: Shuffle row groups and/or rows

In [5]:
# Shuffle row groups (parquet fragments) and rows within batches
dataset_shuffled = PyArrowParquetDataset(
    data_path,
    batch_size=8,
    shuffle_row_groups=True,  # Shuffle order of parquet files
    shuffle_rows=True,         # Shuffle rows within each batch
    seed=42                    # For reproducibility
)

loader_shuffled = DataLoader(dataset_shuffled, batch_size=None)

# Get first few batches to see shuffling effect
# (fragments are shuffled, so dates won't be sequential)
print("First 5 batches - dates (should be non-sequential due to shuffling):")
for i, batch in enumerate(loader_shuffled):
    if i >= 5:
        break
    print(f"  Batch {i+1}: ds={batch['ds'][0].item()}, h={batch['h'][0].item()}")

First 5 batches - dates (should be non-sequential due to shuffling):
  Batch 1: ds=20260114, h=13
  Batch 2: ds=20260114, h=13
  Batch 3: ds=20260114, h=13
  Batch 4: ds=20260114, h=13
  Batch 5: ds=20260114, h=13


## Multi-worker DataLoader Support

In [6]:
# Test multi-worker DataLoader (fragments are automatically sharded across workers)
dataset_multi = PyArrowParquetDataset(
    data_path,
    batch_size=8,
    seed=42
)

# Use 2 workers - each worker gets a subset of fragments
loader_multi = DataLoader(
    dataset_multi, 
    batch_size=None,
    num_workers=2,
    pin_memory=False  # Set to True if using GPU
)

# Get batches from multiple workers
print("Testing multi-worker DataLoader...")
for i, batch in enumerate(loader_multi):
    if i >= 5:  # Just show first 5 batches
        break
    print(f"Batch {i+1}: {len(batch['ds'])} rows, dates {batch['ds'].min().item()}-{batch['ds'].max().item()}")

Testing multi-worker DataLoader...


/Users/fox/Projects/jupyter_notebook_projects/ml_misc/torch_data_pipe_learning/pyarrow_dataset.py:129: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:209.)
  key: torch.as_tensor(value)
/Users/fox/Projects/jupyter_notebook_projects/ml_misc/torch_data_pipe_learning/pyarrow_dataset.py:129: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be sup

Batch 1: 8 rows, dates 20260101-20260101
Batch 2: 8 rows, dates 20260101-20260101
Batch 3: 8 rows, dates 20260101-20260101
Batch 4: 8 rows, dates 20260101-20260101
Batch 5: 8 rows, dates 20260101-20260101


## Null Value Detection and Transformation

Handle null values in the data by detecting them and converting to PyTorch tensors with proper transformations.

In [ ]:
# Read data and check for null values
import torch
import numpy as np

# Create dataset
data_path = Path("data").resolve()
dataset = PyArrowParquetDataset(data_path, batch_size=1024)
loader = DataLoader(dataset, batch_size=None)

# Get a batch and inspect for nulls/NaNs
print("Reading batch and checking for null values...\n")
batch = next(iter(loader))

# Check for NaN values in feature columns (PyArrow converts nulls to NaN for float columns)
feature_cols = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']
print("Null/NaN detection in feature columns:")
print("-" * 60)
for col in feature_cols:
    if col in batch:
        tensor = batch[col]
        nan_count = torch.isnan(tensor).sum().item()
        total_count = tensor.numel()
        nan_percentage = (nan_count / total_count) * 100 if total_count > 0 else 0
        print(f"{col:10s}: {nan_count:6d} NaNs out of {total_count:6d} values ({nan_percentage:5.2f}%)")

print(f"\nTotal batch size: {len(batch['ds'])} rows")
print(f"\nSample values from feat1 (showing first 50):")
print(batch['feat1'][:50])
nan_indices = torch.isnan(batch['feat1']).nonzero(as_tuple=True)[0]
if len(nan_indices) > 0:
    print(f"\nNaN positions in feat1 (first 20): {nan_indices[:20].tolist()}")
else:
    print(f"\nNo NaN values found in feat1 for this batch")

Reading batch and checking for null values...

Null/NaN detection in feature columns:
------------------------------------------------------------
feat1     :     28 NaNs out of    500 values ( 5.60%)
feat2     :     25 NaNs out of    500 values ( 5.00%)
feat3     :     25 NaNs out of    500 values ( 5.00%)
feat4     :     29 NaNs out of    500 values ( 5.80%)
feat5     :     24 NaNs out of    500 values ( 4.80%)

Total batch size: 500 rows

Sample values from feat1 (showing first 20):
tensor([-8.4572e-02,  7.2495e-01,  9.3491e-02, -1.1296e+00,  1.7600e+00,
        -1.4237e-01,  1.0191e+00,  4.7302e-01,  1.3003e-03, -2.9866e+00,
        -5.6973e-01, -2.6545e-01, -1.5202e+00, -6.0842e-01,  1.9037e+00,
         8.8043e-01,  2.9179e+00, -1.1246e-03,  8.1477e-01,  1.4406e+00,
         1.1798e-02, -8.9396e-01, -1.4583e+00,  1.7694e+00,  1.7061e-01,
         1.3209e+00,  4.7767e-01,  5.5893e-02, -6.9187e-01, -3.2147e+00,
        -8.5261e-01, -1.1991e+00,  9.4817e-02,  1.7161e-01, -5.4921e-01

In [5]:
# Define transformation function to handle null values
def transform_batch(batch, fill_value=0.0, feature_cols=None):
    """
    Transform batch to handle null/NaN values.
    
    Args:
        batch: Dictionary of tensors
        fill_value: Value to fill NaN with (default: 0.0)
        feature_cols: List of feature column names to transform (default: all feat* columns)
    
    Returns:
        Transformed batch dictionary
    """
    if feature_cols is None:
        feature_cols = [col for col in batch.keys() if col.startswith('feat')]
    
    transformed_batch = batch.copy()
    
    for col in feature_cols:
        if col in transformed_batch:
            tensor = transformed_batch[col]
            # Replace NaN with fill_value
            transformed_batch[col] = torch.where(
                torch.isnan(tensor),
                torch.tensor(fill_value, dtype=tensor.dtype, device=tensor.device),
                tensor
            )
    
    return transformed_batch

# Test the transformation
print("Testing transformation function...\n")
batch_with_nulls = next(iter(loader))
print(f"Before transformation - NaNs in feat1: {torch.isnan(batch_with_nulls['feat1']).sum().item()}")

transformed_batch = transform_batch(batch_with_nulls, fill_value=0.0)
print(f"After transformation - NaNs in feat1: {torch.isnan(transformed_batch['feat1']).sum().item()}")
print(f"✓ All NaNs replaced with 0.0")

Testing transformation function...

Before transformation - NaNs in feat1: 28
After transformation - NaNs in feat1: 0
✓ All NaNs replaced with 0.0


In [6]:
# Create a wrapper dataset that automatically handles nulls
from torch.utils.data import IterableDataset

class NullHandlingDataset(IterableDataset):
    """
    Wrapper around PyArrowParquetDataset that automatically handles null values.
    """
    def __init__(self, base_dataset, fill_value=0.0, feature_cols=None):
        """
        Args:
            base_dataset: PyArrowParquetDataset instance
            fill_value: Value to fill NaN with (default: 0.0)
            feature_cols: List of feature column names to transform
        """
        self.base_dataset = base_dataset
        self.fill_value = fill_value
        self.feature_cols = feature_cols or [col for col in base_dataset.schema.names if col.startswith('feat')]
    
    def __iter__(self):
        for batch in self.base_dataset:
            yield transform_batch(batch, fill_value=self.fill_value, feature_cols=self.feature_cols)
    
    def __len__(self):
        # IterableDataset doesn't have a length, but we can access base dataset properties
        return getattr(self.base_dataset, '__len__', lambda: None)()

# Create dataset with automatic null handling
dataset_clean = NullHandlingDataset(
    PyArrowParquetDataset(data_path, batch_size=1024),
    fill_value=0.0
)

loader_clean = DataLoader(dataset_clean, batch_size=None)

# Verify null handling works
print("Testing NullHandlingDataset...\n")
clean_batch = next(iter(loader_clean))

print("Null/NaN detection after transformation:")
print("-" * 60)
for col in ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']:
    if col in clean_batch:
        tensor = clean_batch[col]
        nan_count = torch.isnan(tensor).sum().item()
        print(f"{col:10s}: {nan_count:6d} NaNs (should be 0)")

print(f"\n✓ All batches from this dataset will have nulls filled with 0.0")
print(f"✓ Ready for training!")

Testing NullHandlingDataset...

Null/NaN detection after transformation:
------------------------------------------------------------
feat1     :      0 NaNs (should be 0)
feat2     :      0 NaNs (should be 0)
feat3     :      0 NaNs (should be 0)
feat4     :      0 NaNs (should be 0)
feat5     :      0 NaNs (should be 0)

✓ All batches from this dataset will have nulls filled with 0.0
✓ Ready for training!


## Verify: Works with all 1344 files (no hanging!)

In [7]:
# Verify it works with all files instantly
import time

print("Testing with all 1344 files...")
start_time = time.time()

dataset_all = PyArrowParquetDataset(data_path, batch_size=1024)
loader_all = DataLoader(dataset_all, batch_size=None)

# Count total rows across first few batches
total_rows = 0
batch_count = 0
for batch in loader_all:
    total_rows += len(batch['ds'])
    batch_count += 1
    if batch_count >= 10:  # Just check first 10 batches
        break

elapsed = time.time() - start_time
print(f"✓ Processed {batch_count} batches ({total_rows} rows) in {elapsed:.2f} seconds")
print(f"✓ No hanging - works perfectly with all partitions!")

Testing with all 1344 files...
✓ Processed 10 batches (5000 rows) in 0.02 seconds
✓ No hanging - works perfectly with all partitions!
